In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

ModuleNotFoundError: No module named 'sentence_transformers'

In [5]:
embedding = embedding_model.encode(
    "AI will transform software engineering"
)

print(len(embedding))

NameError: name 'embedding_model' is not defined

In [12]:
import chromadb
from sentence_transformers import SentenceTransformer
from app.personas import BOT_PERSONAS



d:\me\CODING\Github\Cognitive-Routing-RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
#Setup ChromaDB and SentenceTransformer
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(
    name="bot_personas",
    metadata={"hnsw:space": "cosine"}
)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4057.00it/s]


In [14]:
def upsert_personas():
    """Upsert bot personas into ChromaDB collection with their embeddings."""
    for bot in BOT_PERSONAS:
        embedding = embedding_model.encode(bot["persona"]).tolist()
        collection.upsert(
            ids=[bot["bot_id"]],
            documents=[bot["persona"]],
            embeddings=[embedding],
            metadatas=[{"name": bot["name"]}],
        )

In [ ]:
def route_post_to_bots(post_content: str, threshold: float = 0.0):
    """Route a post to the most relevant bots based on cosine similarity."""
    post_embedding = embedding_model.encode(post_content).tolist()
    results = collection.query(
        query_embeddings=[post_embedding],
        n_results=3,
    )
    matches = []
    for idx, distance in enumerate(results["distances"][0]):
        similarity = 1 - distance  # Convert cosine distance to similarity
        # if similarity >= threshold:
         matches.append((results["ids"][0][idx], results["metadatas"][0][idx]["name"], similarity))
    return matches

In [38]:
upsert_personas()
tests = [
    "I think AI will revolutionize the world and we should embrace it. However, we should also be cautious about potential risks and ethical concerns.",
    "Tech monopolies are exploiting user data while social media platforms destroy attention spans and personal privacy.",
    "With the onset of AI, we can see the RoI in software development skyrocketing, with huge potential for economy to grow.",
]

for test_post in tests:
    print(route_post_to_bots(test_post))


[('Bot_A', 'Tech Maximalist', 0.3724782466888428)]
[('Bot_B', 'Doomer / Skeptic', 0.0529787540435791)]
[]
